# Pipeline: On-the-Fly Dataset Generation + Sequential Student Fine-Tuning

This notebook generates synthetic data from documents and uses it to sequentially fine-tune a student model. It avoids saving large datasets by training in-memory after each document is processed.

**Flow:**
1.  **Load Models:** Load the 4-bit Teacher (Llama-3.2 3B), the 4-bit base Student (Qwen2 0.5B), and the embedder.
2.  **Loop per Document:**
    a.  Load and chunk the document (e.g., *The Republic*).
    b.  **Generate:** Use the Teacher model to create `N` high-quality (prompt, completion) pairs based on the doc.
    c.  **Train:** Fine-tune the *current best* Student model on these pairs, creating new LoRA adapters.
    d.  **Evaluate:** Run a quick evaluation. 
    e.  **Commit:** If the score improves, push the new adapters to the Hub and set it as the new 'best'.
3.  **Final Merge:** Merge the *final* set of adapters with the base student model and push a standalone, inference-ready model.

## 1. Setup: Installs & Login

In [1]:
# Run this cell once to install all required dependencies with compatible versions
# Note: You may need to restart the kernel after installation
%pip install -q --upgrade pip
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -q transformers==4.46.3 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.1.1
%pip install -q datasets trl sentence-transformers faiss-cpu huggingface_hub tqdm
%pip install -q requests pypdf2 beautifulsoup4 newspaper3k langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.25.0 requires accelerate>=1.4.0, but you have accelerate 1.1.1 which is incompatible.
trl 0.25.0 requires transformers>=4.56.1, but you have transformers 4.46.3 which is incompatible.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login

# ⚠️ You must set the HF_TOKEN environment variable in your runtime.
token = os.getenv("HF_TOKEN")
if token is None:
    raise ValueError("HF_TOKEN environment variable not set! Login required to push models.")

login(token=token)
print("Logged in successfully!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in successfully!


## 2. Configuration

In [3]:
# Imports
import os
import json
import random
import re
import gc
import shutil
import numpy as np
import faiss
import torch
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, pipeline
)
from peft import LoraConfig, PeftModel, get_peft_model
from trl.trainer.sft_trainer import SFTTrainer
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import HfApi, Repository
from tqdm.notebook import tqdm
import requests
from bs4 import BeautifulSoup
import PyPDF2

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, Mem: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

PyTorch: 2.5.1+cu121, CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU, Mem: 8.6GB


In [4]:
# --- Model Configuration ---
# ⚠️ CHANGE THIS to your Hugging Face repo ID (e.g., "YourUser/YourRepo")
HF_REPO_ID = "penguneedLLM/digitalbookllm-student"

# Teacher Model (Smaller, 3B) - Used for generating data and judging
# Switched to 3B to fit in available memory (originally was 8B)
TEACHER_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# Student Model (Smaller, 0.5B) - The model we are fine-tuning
STUDENT_MODEL = "Qwen/Qwen2-0.5B-Instruct"

# Embedder (Lightweight) - Used for RAG retrievals
EMBED_MODEL = "all-MiniLM-L6-v2"

# --- Training Hyperparameters ---
# Score from LLM-as-judge to keep a generated example
SCORE_THRESHOLD = 4.0
# Examples to generate per document (Lowered for resources)
EXAMPLES_PER_DOC = 150
# Batches per device (Lowered for resources)
BATCH_SIZE = 1
# Accumulate gradients to offset small batch size
GRAD_ACCUM_STEPS = 8
# Epochs to train on each document's synthetic dataset
EPOCHS_PER_DOC = 1
# Chunks to retrieve for RAG context
TOP_K = 3

# --- Quantization & LoRA Config ---
# 4-bit config for loading models efficiently
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
    llm_int8_enable_fp32_cpu_offload=True  # Enable FP32 CPU offload for modules that need it
)

# LoRA config for fine-tuning the student
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    # Target modules for Qwen2 0.5B
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none", task_type="CAUSAL_LM"
)

# --- Training Arguments ---
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, 'is_bf16_supported', lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision selection -> bf16: {use_bf16}, fp16: {use_fp16}")

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=2e-4, 
    num_train_epochs=EPOCHS_PER_DOC,
    logging_steps=10, 
    save_strategy="no",
    output_dir="./tmp_trainer_output", 
    optim="paged_adamw_32bit",
    bf16=use_bf16, 
    fp16=use_fp16, 
    max_grad_norm=0.3, 
    warmup_ratio=0.03,
    group_by_length=True, 
    report_to=None  # Disable wandb/comet
)

# --- Document Sources ---
DOC_SOURCES = [
    {"name": "the_republic", "url": "https://www.gutenberg.org/files/1497/1497-0.txt", "field": "Philosophy"},
    {"name": "leviathan", "url": "https://www.gutenberg.org/files/3207/3207-0.txt", "field": "Political Philosophy"},
    {"name": "meditations", "url": "https://www.gutenberg.org/files/2680/2680-0.txt", "field": "Stoicism"},
    {"name": "roman_republic", "url": "https://en.wikipedia.org/wiki/Roman_Republic", "field": "History", "type": "wiki"},
    # Add more URLs (Gutenberg, Wikipedia) here
]

print(f"Config: Teacher={TEACHER_MODEL}, Student={STUDENT_MODEL}, Examples/Doc={EXAMPLES_PER_DOC}")
print(f"HF Repo: {HF_REPO_ID}")

Precision selection -> bf16: True, fp16: False
Config: Teacher=meta-llama/Llama-3.2-3B-Instruct, Student=Qwen/Qwen2-0.5B-Instruct, Examples/Doc=150
HF Repo: penguneedLLM/digitalbookllm-student
Config: Teacher=meta-llama/Llama-3.2-3B-Instruct, Student=Qwen/Qwen2-0.5B-Instruct, Examples/Doc=150
HF Repo: penguneedLLM/digitalbookllm-student


## 3. Load Models (One-Time)

In [5]:
# Load teacher (once, for gen/judging)
print("Loading teacher model... (Llama-3.2 3B)")
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

# Load embedder (lightweight)
print("Loading embedder model...")
embedder = SentenceTransformer(EMBED_MODEL)

# Load base student model (once, for fine-tuning)
# We load this *once* to save VRAM, rather than in the loop.
print("Loading base student model... (Qwen2 0.5B)")
base_student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
student_tokenizer.pad_token = student_tokenizer.eos_token

print("All models loaded.")

Loading teacher model... (Llama-3.2 3B)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading embedder model...
Loading base student model... (Qwen2 0.5B)
Loading base student model... (Qwen2 0.5B)
All models loaded.
All models loaded.


## 4. Helper Functions

### 4.1. Data Loading and RAG

In [6]:
# Doc loading helpers
def load_text_from_url(url: str, doc_type: str = "text") -> str:
    """Loads and cleans text from Gutenberg or Wikipedia."""
    try:
        if doc_type == "wiki":
            resp = requests.get(url)
            soup = BeautifulSoup(resp.content, 'html.parser')
            text = ' '.join([p.text for p in soup.find_all('p')])
        else:  # Gutenberg TXT
            resp = requests.get(url)
            text = resp.text
            # Clean Gutenberg header/footer
            start = text.find("*** START")
            end = text.find("*** END")
            if start != -1 and end != -1:
                text = text[start+10:end].strip()
        return text
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return ""

def chunk_text(text: str, chunk_size=1000, overlap=150) -> List[str]:
    """Splits a long text into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return splitter.split_text(text)

def build_index(chunks: List[str]) -> Optional[Tuple[faiss.Index, Dict[int, str]]]:
    """Creates a FAISS vector index for the text chunks."""
    if not chunks:
        return None
    embeddings = embedder.encode(chunks, show_progress_bar=False)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype('float32'))
    chunk_map = {i: chunk for i, chunk in enumerate(chunks)}
    return index, chunk_map

# RAG helpers
def retrieve_chunks(index: faiss.Index, chunk_map: Dict, query: str, top_k: int) -> List[str]:
    """Retrieves the top_k most relevant chunks for a query."""
    query_emb = embedder.encode([query])
    _, indices = index.search(query_emb.astype('float32'), top_k)
    return [chunk_map[i] for i in indices[0]]

print("Data loading/RAG helpers loaded.")

Data loading/RAG helpers loaded.


### 4.2. Synthetic Data Generation (using Teacher)

In [7]:
# Prompts for the Teacher Model
def build_search_prompt(query: str, contexts: List[str]) -> str:
    context = "\n\n".join(contexts)
    return f"Answer based only on the context: {context}\n\nQuestion: {query}\nAnswer:"

def build_tutor_prompt(query: str, selected_text: str, related: List[str]) -> str:
    related = "\n\n".join(related)
    return f"User selected: {selected_text}\nRelated: {related}\n\nSynthesize a conversational answer to: {query}\nAnswer:"

# Generation strategies
def generate_question_from_chunk(chunk: str) -> str:
    """(Strategy 0) Generate a question that this chunk answers."""
    prompt = f"""Generate a single, natural-sounding question that the following text answers. Do not answer it.

Text: {chunk[:800]}...

Question:"""
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=60, temperature=0.8, do_sample=True)
    return teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Question:")[-1].strip()

def generate_search_mode_pair(index, chunk_map, all_chunks) -> Tuple[str, str]:
    """(Strategy 1: Search) Simulate a user asking a question, answered with RAG."""
    chunk = random.choice(all_chunks)
    query = generate_question_from_chunk(chunk)
    if not query:
        return None, None
    contexts = retrieve_chunks(index, chunk_map, query, TOP_K)
    prompt = build_search_prompt(query, contexts)
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=250, temperature=0.2, do_sample=True)
    completion = teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[-1].strip()
    return prompt, completion

def generate_tutor_ask_pair(index, chunk_map, all_chunks) -> Tuple[str, str]:
    """(Strategy 2: Tutor) Simulate user highlighting text and asking a question."""
    selected = random.choice(all_chunks)
    query = f"Explain this in simpler terms: {selected[:150]}..."
    augmented_query = query + " " + selected # Use full text for retrieval
    related = retrieve_chunks(index, chunk_map, augmented_query, TOP_K)
    related = [r for r in related if r != selected] # Dedup
    prompt = build_tutor_prompt(query, selected, related)
    inputs = teacher_tokenizer(prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=250, temperature=0.2, do_sample=True)
    completion = teacher_tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[-1].strip()
    return prompt, completion

strategies = [generate_search_mode_pair, generate_tutor_ask_pair]

# LLM-as-Judge
def judge_response(prompt: str, completion: str) -> float:
    """Uses the teacher model to score the (prompt, completion) pair."""
    judge_prompt = f"""Rate the quality of the 'completion' as an answer to the 'prompt' on a scale of 0.0 to 5.0.
Consider clarity, relevance, and faithfulness to the likely context.
Respond with *only* the float score (e.g., 4.5).

Prompt: {prompt[:600]}...
Completion: {completion[:300]}...

Score:"""
    inputs = teacher_tokenizer(judge_prompt, return_tensors="pt").to(teacher_model.device)
    out = teacher_model.generate(**inputs, max_new_tokens=10, temperature=0.1, do_sample=False)
    text = teacher_tokenizer.decode(out[0], skip_special_tokens=True)
    # Find the first float in the response (e.g., "Score: 4.5" -> 4.5)
    match = re.search(r'([0-5]\.[0-9])', text)
    return float(match.group(1)) if match else 0.0

print("Generation/Judging functions ready.")

Generation/Judging functions ready.


### 4.3. Student Training & Evaluation

In [ ]:
def load_student_adapters(base_model: AutoModelForCausalLM, adapter_path: str) -> PeftModel:
    """
    Loads adapters onto the (already-loaded) base student model.
    This is the core of sequential training:
    1. Load adapters from `adapter_path`.
    2. Merge them into the base model.
    3. Apply *new, fresh* adapters for the *next* training run.
    """
    if adapter_path == STUDENT_MODEL:  # Base case (first run)
        print("Applying fresh LoRA adapters to base student model.")
        model = get_peft_model(base_model, lora_config)
    else:  # Subsequent runs
        print(f"Loading adapters from {adapter_path}, merging, and applying new adapters.")
        # 1. Load previous best adapters
        model = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=True)
        # 2. Merge them into the model (e.g., 4-bit base + adapters = new 4-bit model)
        model = model.merge_and_unload()
        # 3. Add new, empty adapters for this training run
        model = get_peft_model(model, lora_config)
    return model

def evaluate_model(model, tokenizer) -> float:
    """Stub evaluation. Replace with a real, robust test set."""
    test_prompts = [
        "What is Stoicism?", 
        "Explain the concept of the 'Leviathan' in political philosophy."
    ]
    scores = []
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")
    for p in test_prompts:
        # Use the Qwen2 prompt format for eval
        formatted_prompt = f"<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n"
        out = pipe(formatted_prompt, max_new_tokens=50, do_sample=False)[0]['generated_text']
        # Dummy score: just check if it generated *something*
        response = out.split("<|im_start|>assistant\n")[-1].strip()
        if response.endswith("<|im_end|>"):
            response = response[:-len("<|im_end|>")].strip()
        scores.append(len(response) / 50.0) # Score 1.0 if it hits 50 tokens
    return np.mean(scores)

def push_lora_to_hub(repo_id: str, local_dir: str, bitmask: int, doc_idx: int) -> str:
    """Pushes the contents of a local adapter dir to a unique HF repo path."""
    api = HfApi()
    # Create a unique repo name for this specific set of adapters
    adapter_repo_path = f"{repo_id}/adapters_doc_{doc_idx}_bitmask_{bitmask:04b}"
    
    print(f"Pushing new adapters to: {adapter_repo_path}")
    # Save tokenizer config (needed by peft)
    student_tokenizer.save_pretrained(local_dir)
    
    # Upload the entire directory (adapter_config.json, adapter_model.bin, etc.)
    api.upload_folder(
        folder_path=local_dir,
        repo_id=adapter_repo_path,
        repo_type="model",
        commit_message=f"Update LoRA: doc={doc_idx} ('{DOC_SOURCES[doc_idx]['name']}'), bitmask={bitmask:04b}"
    )
    
    print(f"Successfully pushed to {adapter_repo_path}")
    # This HF repo ID is now the 'path' to our best adapters
    return adapter_repo_path

print("Student helper functions ready.")

Student helper functions ready.


## 5. Main Loop: Per-Document Generation + Training

In [9]:
# Helper to format data for SFTTrainer
def format_for_sft(pairs: List[Dict]) -> Dataset:
    """Use the Qwen2 chat template format:
    <|im_start|>user
    {prompt}<|im_end|>
    <|im_start|>assistant
    {completion}<|im_end|>
    """
    texts = [
        f"<|im_start|>user\n{p['prompt']}<|im_end|>\n<|im_start|>assistant\n{p['completion']}<|im_end|>"
        for p in pairs
    ]
    return Dataset.from_dict({"text": texts})

# --- Main Loop ---

# `best_path` tracks the HF repo ID of the best adapters found so far
best_path = STUDENT_MODEL  # Start with the base model ID
best_score = 0.0
trained_bitmask = 0
num_docs = len(DOC_SOURCES)

# 1. Initial Evaluation (of the base model)
print("Evaluating base student model...")
temp_model_for_eval = load_student_adapters(base_student_model, best_path)
best_score = evaluate_model(temp_model_for_eval, student_tokenizer)
del temp_model_for_eval
gc.collect(); torch.cuda.empty_cache()
print(f"Base model score: {best_score:.3f}")

# 2. Loop through each document
for i, source in enumerate(DOC_SOURCES):
    print(f"\n{'='*60}")
    print(f"Processing Doc {i+1}/{num_docs}: {source['name']} ({source['field']})")
    print(f"{'='*60}")
    
    # 2a. Load doc and build index
    text = load_text_from_url(source['url'], source.get('type', 'text'))
    chunks = chunk_text(text)
    index_data = build_index(chunks)
    if not index_data:
        print("Failed to load or chunk text. Skipping doc.")
        continue
    index, chunk_map = index_data
    
    # 2b. Generate synthetic data in-memory
    high_quality_pairs = []
    pbar = tqdm(range(EXAMPLES_PER_DOC), desc=f"Gen (Doc {i+1})")
    for _ in pbar:
        strat = random.choice(strategies)
        prompt, comp = strat(index, chunk_map, chunks)
        if prompt and comp:
            score = judge_response(prompt, comp)
            pbar.set_postfix(score=f"{score:.2f}", kept=len(high_quality_pairs))
            if score >= SCORE_THRESHOLD:
                high_quality_pairs.append({"prompt": prompt, "completion": comp})
    
    print(f"Generated {len(high_quality_pairs)} high-quality pairs (from {EXAMPLES_PER_DOC} attempts).")
    
    if len(high_quality_pairs) < 10:  # Not enough data to train
        print("Skipping training, not enough high-quality data.")
        continue
    
    # 2c. Train student on new data
    print("Loading student model for training...")
    train_dataset = format_for_sft(high_quality_pairs)
    # This re-uses base_student_model, loads `best_path` adapters, merges, and adds new ones
    model = load_student_adapters(base_student_model, best_path)
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=student_tokenizer,
        train_dataset=train_dataset,
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=2048, # Adjust as needed
        args=training_args
    )
    
    print("Starting training...")
    trainer.train()
    
    # 2d. Save new adapters and evaluate
    temp_dir = f"./temp_lora_doc_{i}"
    trainer.model.save_pretrained(temp_dir) # Saves *only* the new adapters
    
    print("Evaluating new model...")
    # Merge the new adapters for evaluation
    model = model.merge_and_unload()
    new_score = evaluate_model(model, student_tokenizer)
    print(f"New score: {new_score:.3f} (vs best {best_score:.3f})")
    
    # 2e. Commit or Rollback
    current_bit = 1 << i
    new_mask = trained_bitmask | current_bit
    
    if new_score > best_score:
        print("✓ Improvement! Pushing new adapters to Hub...")
        best_path = push_lora_to_hub(HF_REPO_ID, temp_dir, new_mask, i)
        best_score = new_score
        trained_bitmask = new_mask
        print(f"New best_path: {best_path}")
    else:
        print("✗ No improvement. Rolling back adapters.")
    
    # 2f. Cleanup
    del model, trainer, train_dataset
    shutil.rmtree(temp_dir, ignore_errors=True) # Delete local temp adapters
    gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"Final Run Complete!")
print(f"Final Best Adapter Path: {best_path}")
print(f"Final Trained Bitmask: {trained_bitmask:0{num_docs}b}")
print(f"Final Best Score: {best_score:.3f}")
print(f"{'='*60}")

Evaluating base student model...
Applying fresh LoRA adapters to base student model.


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Base model score: 5.470

Processing Doc 1/4: the_republic (Philosophy)


Gen (Doc 1):   0%|          | 0/150 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-

...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Generated 0 high-quality pairs (from 150 attempts).
Skipping training, not enough high-quality data.

Processing Doc 3/4: meditations (Stoicism)


Gen (Doc 3):   0%|          | 0/150 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Generated 0 high-quality pairs (from 150 attempts).
Skipping training, not enough high-quality data.

Processing Doc 4/4: roman_republic (History)
Failed to load or chunk text. Skipping doc.

Final Run Complete!
Final Best Adapter Path: Qwen/Qwen2-0.5B-Instruct
Final Trained Bitmask: 0000
Final Best Score: 5.470
Failed to load or chunk text. Skipping doc.

Final Run Complete!
Final Best Adapter Path: Qwen/Qwen2-0.5B-Instruct
Final Trained Bitmask: 0000
Final Best Score: 5.470


## 6. Final Merge and Push

In [10]:
# This final step creates a standalone, inference-ready model.
# It loads the *original* student model (in float16) and merges the
# *final best* adapters into it.

if best_path == STUDENT_MODEL:
    print("No training was successful. Skipping final merge.")
else:
    print("Merging final model for inference...")
    # Load base in float16 for merging (not 4-bit)
    base = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL, 
        device_map="auto", 
        torch_dtype=torch.float16, 
        trust_remote_code=True
    )
    
    print(f"Loading final best adapters from: {best_path}")
    final_model = PeftModel.from_pretrained(base, best_path)
    final_model = final_model.merge_and_unload()
    
    final_path = f"{HF_REPO_ID}/final_merged_model"
    print(f"Pushing merged model to {final_path}...")
    
    final_model.push_to_hub(final_path, commit_message="Push final merged model")
    student_tokenizer.push_to_hub(final_path, commit_message="Push tokenizer for final model")
    
    print(f"Pushed merged model to {final_path}")

# Cleanup all models
del teacher_model, base_student_model, embedder
gc.collect(); torch.cuda.empty_cache()

No training was successful. Skipping final merge.


## 7. Next Steps

-   **Test Inference**: Load your final model from the Hub: `AutoModelForCausalLM.from_pretrained("YOUR_HF_REPO/final_merged_model")`.
-   **Expand Eval**: The `evaluate_model` function is a stub. Create a real, hold-out test set of prompts and use a better metric (like ROUGE or a separate LLM-as-judge) for a true measure of quality.
-   **Scale Docs**: Add more `DOC_SOURCES`. You can also adapt the `load_text_from_url` function to handle local PDF uploads (e.g., for OpenStax books) if you're running this in an environment where you can upload files.